In [ ]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F
import math


In [49]:
import urllib.request
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Download Tiny Shakespeare
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, 'shakespeare.txt')

with open('shakespeare.txt', 'r') as f:
    text = f.read()

# 2. Simple Character-Level Tokenizer
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Dictionaries to map characters to integers and back
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# Encoder and Decoder functions
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Convert entire dataset to a PyTorch tensor
data = torch.tensor(encode(text), dtype=torch.long)

# 3. Create a Custom PyTorch Dataset
class CharDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len
        
    def __len__(self):
        # We subtract seq_len so we don't go out of bounds at the end of the text
        return len(self.data) - self.seq_len
        
    def __getitem__(self, idx):
        # Input sequence (Context Window)
        x = self.data[idx : idx + self.seq_len]
        
        # Target sequence (Shifted right by 1 character to predict the next token)
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        
        return x, y

# 4. Create the PyTorch DataLoader
seq_len = 2048    # Your context window
batch_size = 2048  # How many sequences to process at once

dataset = CharDataset(data, seq_len)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# --- Test the DataLoader ---
x_batch, y_batch = next(iter(dataloader))
print("Dataset Loaded Successfully!")
print(f"Vocab size: {vocab_size}")
print(f"x_batch shape: {x_batch.shape}")
print(f"y_batch shape: {y_batch.shape}")

Dataset Loaded Successfully!
Vocab size: 65
x_batch shape: torch.Size([2048, 2048])
y_batch shape: torch.Size([2048, 2048])


In [ ]:
device = "cuda: 0"

In [4]:
class Linear(nn.Module): 
  def __init__(self, d_in= 512, d_out=1024, last = False):
        super().__init__()
        self.W = nn.Parameter(
                        torch.randn(
                                size = (d_in, d_out),
                                requires_grad=True, 
                                dtype = torch.bfloat16
                        ))
        self.b = nn.Parameter(
                        torch.zeros(
                                size = (d_out, ),
                                requires_grad=True, 
                                dtype = torch.bfloat16
                        ))

        self.last = last
  def act(self, x): 
    if self.last: 
      return F.relu(x)
    else: 
      return x
     
  def forward(self, x): 
    #x -> (B, d_in) @ ( d_in, d_out)
    return self.act(torch.matmul(x, self.W ) + self.b)

In [5]:
class Embedding(nn.Module): 
    def __init__(self, d_in= 10000, d_out=512):
        super().__init__()
        self.W = nn.Parameter(
                        torch.randn(
                                size = (d_in, d_out),
                                requires_grad=True, 
                                dtype = torch.bfloat16
                        ))
        self.d_in = d_in
    def forward(self, x): 
        #encode in one hot to access embedding vector
        x= F.one_hot(x, num_classes=self.d_in).to(torch.bfloat16)
        #(batch, v) x (v, d_model)
        return x.matmul(self.W)


In [6]:
from numpy import True_
class MLP(nn.Module): 
  def __init__(self, d_model=512):
        super().__init__()
        self.L1 = Linear(d_in =d_model, d_out = d_model*2)
        self.L2 = Linear(d_in = d_model*2, d_out = d_model*2)
        self.L3 = Linear(d_in = d_model*2, d_out = d_model*4)
        self.L4 = Linear(d_in = d_model*4, d_out = d_model*2)
        self.L5 = Linear(d_in = d_model*2, d_out = d_model, last = True_)
        self.layers = nn.ModuleList([
          self.L1,
          self.L2,
          self.L3,
          self.L4,
          self.L5,
          ])
  def forward(self, x): 
    for l in self.layers: 
      x = l(x)
    return x
    



#pos enc is ai generated 

In [40]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        
        # Create a matrix of shape (max_len, d_model) filled with zeros
        pe = torch.zeros(max_len, d_model)
        
        # Create a vector of positions (0, 1, 2, ...) and reshape to (max_len, 1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Calculate the division term (denominator) for the sine and cosine formulas
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # Apply sine to even indices in the array
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # Apply cosine to odd indices in the array
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Register as buffer so it's saved with the model state but not treated as a trainable parameter
        # Cast the float32 math into bfloat16 before saving it!
        self.register_buffer('pe', pe.unsqueeze(0).to(torch.bfloat16))


    def forward(self, x):
        # x shape expected: (batch_size, seq_len, d_model)
        # Add the positional encoding to the input embeddings
        x = x + self.pe[:, :x.size(1)]
        return x


In [41]:
class LayerNorm(nn.Module):
    def __init__(self,  d_model=512, eps = 1e-5):
        super().__init__()
        self.beta = nn.Parameter(
                                torch.zeros(
                                        size = (d_model, ),
                                        requires_grad=True, 
                                        dtype = torch.bfloat16
                                ) )
        self.gamma = nn.Parameter(
                                torch.ones(
                                        size = (d_model, ),
                                        requires_grad=True, 
                                        dtype = torch.bfloat16
                                ) )
        self.eps = eps
    def forward(self, x): 
        # over non batch
        meu =   torch.mean( x, dim = -1, keepdim = True)
        sigma = torch.var(x, dim = -1, keepdim = True)
        x_hat = (x - meu ) / torch.sqrt(sigma + self.eps)
        return self.gamma * x_hat + self.beta



In [42]:
import torchgen
class MultiHeadAttention(nn.Module): 
    def __init__(self,  d_model=512, n = 8, mask = True):
        super().__init__()
        assert d_model%n ==0 
        self.n = n
        self.d_model = d_model
        self.mask = mask
        self.K = Linear(d_model, d_model, last = True)
        self.Q = Linear(d_model, d_model, last = True)
        self.V = Linear(d_model, d_model, last = True)
        self.O = Linear(d_model, d_model, last = True)

        self.h_dim = d_model//n 
    def forward(self, x):
                B, seq_len, _= x.shape
                K= self.K(x)
                V= self.V(x)
                Q= self.Q(x)
                # x:(B , seq_len, d_model) 
                # q: (B , seq_len, d_model, d_model) - > (B, seq_len, n, h_dim ) 
                K = K.view(B, seq_len, self.n, self.d_model//self.n)
                V = V.view(B, seq_len, self.n, self.d_model//self.n)
                Q = Q.view(B, seq_len, self.n, self.d_model//self.n)
                # q: (B, seq_len, n, h_dim ) - > (B, n, seq_len h_dim) 
                K = K.transpose(1,2)
                V = V.transpose(1,2)
                Q = Q.transpose(1,2)
                # q: (B, n, seq_len, h_dim) x k: (B, n,  h_dim, seq_len)  - > (B, n, seq_len, seq_len)

                scores = Q.matmul(K.transpose(-2, -1)) / math.sqrt(self.d_model// self.n)
                # q: (B, n, seq_len, h_dim) x k: (B, n,  h_dim, seq_len)  - > (B, n, seq_len, seq_len)

                if self.mask:
                    causal_mask = torch.tril(torch.ones(size = (seq_len, seq_len), device = x.device))
                    scores = scores.masked_fill( causal_mask==0, float('-inf') )


                # softmax over position 3 : seq_len
                sm = F.softmax(scores , dim =-1)
                # softmax: (B, n, seq_len, seq_len) x (B, n, seq_len, h_dim) -> (B, n, seq_len, h_dim) 
                attn = sm.matmul(V) 
                # communication is over now we bring it back to orignal x shape 
                #(B, n, seq_len, h_dim) -> (B, seq_len, d_model) 
                # (B, n, seq_len, h_dim) -> (B, seq_len, n,  h_dim) 
                x = attn.transpose(1,2)
                # 0    1     2      3 
                #(B, seq_len, n,  h_dim)  -> (B, seq_len, d_model) 
                x = x.contiguous().flatten(start_dim = 2, end_dim= 3)

                return self.O(x)

In [43]:
class GPTTransformer(nn.Module): 
        """ 
        """
        def __init__(self, e = 1024 , d_model=512, n = 8, v =65, cw = 2048):
                super().__init__()
                self.pos_enc= PositionalEncoding(max_len=cw, d_model=d_model)
                self.emb = Embedding(d_in = v, d_out= d_model)
                self.masked_attn = MultiHeadAttention(d_model= d_model, n = n)
                self.ln1 = LayerNorm(d_model)
                self.mlp = MLP(d_model)
                self.ln2 = LayerNorm(d_model)     
                self.linear = Linear(d_model, v)

        def forward(self, x): 
                # embed
                x = self.emb(x)
                # pos enc
                x = x + self.pos_enc(x)
                # masked attn
                x = x + self.masked_attn(self.ln1(x))
                # mlp 
                x = x + self.mlp(self.ln2(x))
                # project to vocab
                x = self.linear(x)
                # return raw logits
                return  x


In [44]:
model = GPTTransformer()
model.to("mps")

GPTTransformer(
  (pos_enc): PositionalEncoding()
  (emb): Embedding()
  (masked_attn): MultiHeadAttention(
    (K): Linear()
    (Q): Linear()
    (V): Linear()
    (O): Linear()
  )
  (ln1): LayerNorm()
  (mlp): MLP(
    (L1): Linear()
    (L2): Linear()
    (L3): Linear()
    (L4): Linear()
    (L5): Linear()
    (layers): ModuleList(
      (0-4): 5 x Linear()
    )
  )
  (ln2): LayerNorm()
  (linear): Linear()
)

In [45]:
optimizer = torch.optim.AdamW(lr = 0.001, params= model.parameters())

In [50]:
EPOCHS = 10
lmb = 0.02


for i in range(EPOCHS):
    total_loss = 0
    model.train()

    for batch in dataloader:
        batch_x = batch[0].to("mps")
        batch_y = batch[1].to("mps")
        
        y_pred = model(batch_x) 
        
        loss = F.cross_entropy(y_pred.view(-1, vocab_size), batch_y.view(-1))   

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()
        
        # Add to total loss (.item() pulls the float out of the PyTorch tensor)
        total_loss += loss.item()
        
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {i+1} | Average Loss: {avg_loss:.4f}")


KeyboardInterrupt: 